In [15]:
!pip install requests bs4 pandas tqdm lxml

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 20.2 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 31.9 MB/s eta 0:00:00
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 48.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.3 MB 11.2 MB/s eta 0:00:01
   -------------------------------------- - 11.8/12.3 MB 28.4 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 26.6 MB/s eta 0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
import json
import re
import time


import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

d:\REFERENCE\CHATBOT RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
BASE_URL = "https://cellphones.com.vn"

HEADERS = {
    "authority": "api.sforum.vn",
    "accept": "application/json",
    "accept-language": "en-US,en;q=0.9,vi;q=0.8",
    "content-type": "application/json",
    "origin": "https://cellphones.com.vn",
    "referer": "https://cellphones.com.vn/",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}



In [4]:
def get_links_from_sitemap(sitemap_url, target_keywords=None):
    """
    Hàm đọc sitemap và trả về danh sách các link phù hợp.
    Hỗ trợ cả Sitemap Index (chứa các sitemap khác) và URL Set (chứa link cuối).
    """
    try:
        print(f"Đang quét: {sitemap_url}")
        response = requests.get(sitemap_url, headers=HEADERS, timeout=10)
        
        # Nếu server không trả về 200 OK thì dừng
        if response.status_code != 200:
            print(f"Lỗi {response.status_code} khi truy cập {sitemap_url}")
            return []

        # Parse nội dung XML
        soup = BeautifulSoup(response.content, 'lxml-xml')
        links = set()

        # Trường hợp 1: Đây là Sitemap Index (chứa các sitemap con)
        sitemaps = soup.find_all('sitemap')
        if sitemaps:
            for sitemap in sitemaps:
                loc = sitemap.find('loc')
                if loc:
                    child_sitemap_url = loc.text.strip()
                    # Gọi đệ quy để vào sitemap con. Thêm time.sleep để tránh bị block
                    time.sleep(1) 
                    links.update(get_links_from_sitemap(child_sitemap_url, target_keywords))
            return links

        # Trường hợp 2: Đây là URL Set chứa các link bài viết/sản phẩm
        urls = soup.find_all('url')
        for url in urls:
            loc = url.find('loc')
            if loc:
                link = loc.text.strip()
                
                # Lọc theo từ khóa nếu có (ví dụ: 'laptop', 'iphone', 'samsung')
                if target_keywords:
                    if any(keyword.lower() in link.lower() for keyword in target_keywords):
                        links.add(link)
                else:
                    links.add(link)

        return links

    except Exception as e:
        print(f"Đã xảy ra lỗi tại {sitemap_url}: {e}")
        return []

In [ ]:
if __name__ == "__main__":
    # URL Sitemap gốc của Cellphones (thường là file này hoặc file chứa các sitemap sản phẩm)
    # Lưu ý: Cần kiểm tra file robots.txt của trang để lấy link sitemap chính xác nhất.

    SITEMAP_INDEX = "https://cellphones.com.vn/sitemap/sitemap_index.xml?v=google"
    

    KEYWORDS = ["laptop", "ipad", "apple", "macbook", "iphone", "samsung", "xiaomi", "oppo", "dien-thoai"]
    
    print("Bắt đầu thu thập link...")
    product_links = get_links_from_sitemap(SITEMAP_INDEX, KEYWORDS)
    
    print("\n--- HOÀN THÀNH ---")
    print(f"Tìm thấy tổng cộng {len(product_links)} link phù hợp.")
    
    # Lưu kết quả ra file txt
    with open("cellphones_links.txt", "w", encoding="utf-8") as f:
        for link in sorted(product_links):
            f.write(link + "\n")
            
    print("Đã lưu toàn bộ link vào file 'cellphones_links.txt'.")

Bắt đầu thu thập link...
Đang quét: https://cellphones.com.vn/sitemap/sitemap_index.xml?v=google
Đang quét: https://cellphones.com.vn/sitemap/category-sitemap.xml
Đang quét: https://cellphones.com.vn/sitemap/category-sitemap2.xml
Đang quét: https://cellphones.com.vn/sitemap/category-sitemap3.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap2.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap3.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap4.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap5.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap6.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap7.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap8.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap9.xml
Đang quét: https://cellphones.com.vn/sitemap/product-sitemap10.xml
Đang quét: https://cellphones.com.vn/sitema

In [13]:
with open("cellphones_links.txt", "r", encoding="utf-8") as f:
    for link in f:
        print(link.strip())


https://cellphones.com.vn/action-camera-xiaomi-mi-4k-waterproof-housing.html
https://cellphones.com.vn/am-dun-nuoc-sieu-toc-xiaomi-mi-insulated-kettle-bhr9049gl-cu.html
https://cellphones.com.vn/am-dun-nuoc-sieu-toc-xiaomi-mi-insulated-kettle-bhr9049gl.html
https://cellphones.com.vn/am-dun-nuoc-sieu-toc-xiaomi-mi-smart-kettle-pro-gl-bhr4198gl-cu.html
https://cellphones.com.vn/am-dun-nuoc-sieu-toc-xiaomi-mi-smart-kettle-pro-gl-bhr4198gl.html
https://cellphones.com.vn/am-dun-nuoc-sieu-toc-xiaomi-mi-smart-kettle-zhf4012gl.html
https://cellphones.com.vn/am-dung-nuoc-sieu-toc-xiaomi-electric-kettle-eu-skv4035gl-cu.html
https://cellphones.com.vn/am-dung-nuoc-sieu-toc-xiaomi-electric-kettle-eu-skv4035gl.html
https://cellphones.com.vn/am-sieu-toc-xiaomi-smart-kettle-2-pro.html
https://cellphones.com.vn/apple
https://cellphones.com.vn/apple-12w-usb-power-adapter.html
https://cellphones.com.vn/apple-5w-usb-power-adapter.html
https://cellphones.com.vn/apple-airpod-2-sac-khong-day-cu-99.html
https

In [ ]:
def filter_links(input_file, output_file):
    # Danh sách các từ khóa "rác" cần loại bỏ
    EXCLUDE_KEYWORDS = [
        # Đồ gia dụng, smarthome
        "am-dun-nuoc","am-dung-nuoc", "am-sieu-toc", "may-loc-khong-khi", 
        "robot-hut-bui", "tivi", "may-chieu", "quat", "camera", "nha-thong-minh", 
        "den-ngu", "loa", "lo-vi-song",
        
        # Phụ kiện
        "op-lung","op-lung-cho", "p-lung", "bao-da", "dan-man-hinh", "cuong-luc", "ppf",
        "sac", "cap", "adapter", "power-adapter", "pin-du-phong", 
        "tai-nghe", "earpods", "airpods", "ban-phim", "chuot", 
        "balo", "tui-chong-soc", "the-nho", "usb", "day-deo", 
        "apple-tv", "airpod", "airtag", "pencil", "macmini", 
        "studio", "watch", "but-cam-ung", "cong-chuyen-doi", "imac",
        "cu-dep", "tray-xuoc", "xuoc-can", "da-kich-hoat", "trung-bay",
        "ic-cam-bien", "ic-cam-ung", "ic-nguon", "ic-imei", "ic-o-cung", 
        "ic-song", "ic-wifi", "gia-do", "keyboard"

        # Dịch vụ / Khác
        "thay-man-hinh", "thay-pin", "bao-hanh", "sim-4g", "dan-chong-va-dap", "dich-vu-apple-care", "dich-vu",
    ]

    valid_links = []

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            links = f.readlines()

        for link in links:
            link = link.strip()
            if not link:
                continue

            # 1. Bỏ qua các link không phải là trang sản phẩm chi tiết 
   
            if not link.endswith(".html"):
                continue

            # 2. Bỏ qua nếu link chứa bất kỳ từ khóa nào trong Blacklist
            # Chuyển link về chữ thường (lower) để so sánh không phân biệt hoa thường
            is_valid = True
            for keyword in EXCLUDE_KEYWORDS:
                if keyword in link.lower():
                    is_valid = False
                    break 

            # Nếu thỏa mãn mọi điều kiện thì giữ lại
            if is_valid:
                valid_links.append(link)

        # Ghi kết quả ra file mới
        with open(output_file, 'w', encoding='utf-8') as f:
            for link in valid_links:
                f.write(link + "\n")

        print(f"Đã lọc xong!")
        print(f"Số link ban đầu: {len(links)}")
        print(f"Số link sau khi lọc: {len(valid_links)}")
        print(f"Đã lưu tại file: {output_file}")

    except FileNotFoundError:
        print(f"Không tìm thấy file {input_file}. Hãy kiểm tra lại tên hoặc đường dẫn.")